In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

# --- 1. Rebuild and Load base panel data self-containedly ---
# ROOT = Path.cwd()
# RAW_PATH =  ROOT / 
# print(ROOT)
# RAW_PATH = "data/meft_2020-2025.csv"

CSV_PATH = "/workspaces/healthcareDA/data/meft_2020-2025.csv"


raw = pd.read_csv(CSV_PATH)

# # Standardize column names
column_map = {
    '약효분류코드': 'drug_class',
    '약효분류명': 'drug_class_name',
    '진료년월': 'month',
    '수량_천개': 'quantity',
    '사용금액_백만원': 'amount'
}
panel_raw = raw[list(column_map.keys())].rename(columns=column_map)
panel_raw["drug_class"] = pd.to_numeric(panel_raw["drug_class"], errors="coerce").astype("Int64")
panel_raw["drug_class_name"] = panel_raw["drug_class_name"].astype("string")
panel_raw["month"] = pd.to_datetime(panel_raw["month"], format="%Y-%m", errors="coerce")
panel_raw["amount"] = pd.to_numeric(panel_raw["amount"], errors="coerce")
panel_raw["quantity"] = pd.to_numeric(panel_raw["quantity"], errors="coerce")




In [12]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
약효분류군별 장·단기 성장 비교 분석 + 시각화
------------------------------------------------------------
입력 : meft_2020-2025.csv  (월별 · 약효분류별 수량/사용금액)
출력 : output/ 폴더에
        - growth_metrics.xlsx / .csv    (지표 표)
        - 1_growth_matrix.png           장기 vs 단기 성장 버블 매트릭스
        - 2_yearly_heatmap.png          연도별 성장률 히트맵 (하락 연도 확인)
        - 3_contribution.png            시장 성장 기여도 · 점유율 변화
        - 4_trend_log.png               로그 스케일 추이 (2020=100)
        - 5_risk_return.png             성장 vs 최대낙폭(MDD)

실행 : python atc_growth_analysis.py --csv meft_2020-2025.csv --out output
필요 : pandas, numpy, matplotlib, openpyxl
"""
import argparse
import os
import warnings

import numpy as np
import pandas as pd

%matplotlib inline
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from matplotlib import font_manager as fm
from matplotlib.colors import TwoSlopeNorm
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter, ScalarFormatter

warnings.filterwarnings("ignore")

# ═════════════════════════════════════════════════════════════
# 0. 설정
# ═════════════════════════════════════════════════════════════
UNITS = {  # 지표 컬럼 : (억원 환산용 나눗수, 단위 라벨)
    "사용금액_백만원": (100, "억원"),
    "수량_천개": (1000, "백만개"),
}
EXCLUDE_CODES = [999]      # '해당사항없음' 등 분석 제외 코드
MAX_MISSING_MONTHS = 3     # 결측 월이 이 값보다 많은 분류군은 제외(상위 그룹 행 등)
SHORT_WINDOW = 2           # 단기 CAGR 기간(년)  → 2023→2025 (코로나 기저 이후)
ACCEL_TOL = 0.02           # 단기−장기 격차가 ±2%p 이내면 '유지'
MIN_BASE = 10       # 1억원, CAGR 안정성 기준

C_UP, C_DOWN, C_FLAT = "#2b7bba", "#d1495b", "#9aa5b1"
C_MKT = "#222222"

# def setup_font():
#     cands = ["Malgun Gothic", "AppleGothic", "NanumGothic", "Nanum Gothic",
#              "NanumBarunGothic", "Noto Sans CJK KR", "Noto Sans KR", "Noto Sans CJK JP"]
#     installed = {f.name for f in fm.fontManager.ttflist}
#     for c in cands:
#         if c in installed:
#             plt.rcParams["font.family"] = c
#             break
#     else:
#         print("[경고] 한글 폰트를 찾지 못했습니다. NanumGothic 등을 설치하세요.")
#     plt.rcParams["axes.unicode_minus"] = False
#     plt.rcParams["figure.dpi"] = 100
#     plt.rcParams["axes.spines.top"] = False
#     plt.rcParams["axes.spines.right"] = False


def setup_font():
    try:
        import koreanize_matplotlib  # noqa: F401
    except ImportError:
        pass
    names = {f.name for f in fm.fontManager.ttflist}
    fam = next((f for f in ['NanumGothic', 'Malgun Gothic', 'AppleGothic',
                            'NanumBarunGothic', 'Noto Sans CJK KR', 'Noto Sans CJK JP'] if f in names), None)
    if fam is None:
        raise RuntimeError("한글 폰트 없음: sudo apt-get install -y fonts-nanum 후 커널 재시작")
    # sns.set_theme(style='whitegrid', font=fam, rc={'axes.unicode_minus': False})

    plt.rcParams["axes.unicode_minus"] = False
    plt.rcParams["figure.dpi"] = 100
    plt.rcParams["axes.spines.top"] = False
    plt.rcParams["axes.spines.right"] = False
    plt.rcParams['font.family'] = fam
    return fam


def short_name(s, n=11):
    s = str(s)
    return s if len(s) <= n else s[: n - 1] + "…"


# ═════════════════════════════════════════════════════════════
# 1. 데이터 로드 · 정제
# ═════════════════════════════════════════════════════════════
def load(csv_path, metric):
    # metric = 사용금액(amount), 수량(quantity)
    df = pd.read_csv(csv_path, encoding="utf-8-sig")
    df = df.loc[:, ~df.columns.str.startswith("Unnamed")]
    df["date"] = pd.to_datetime(df["진료년월"] + "-01")
    names = df.drop_duplicates("약효분류코드").set_index("약효분류코드")["약효분류명"]
    monthly = df.pivot(index="date", columns="약효분류코드", values=metric).sort_index()
    return monthly, names


def select_classes(monthly, start, min_base):
    """분석 대상 분류군 선정: 결측 없음 + 기준연도 규모가 너무 작지 않음."""
    n_missing = monthly.isna().sum()
    base = monthly[monthly.index.year == start].sum()
    reason = pd.Series("", index=monthly.columns)
    reason[n_missing > MAX_MISSING_MONTHS] = "결측월 많음(상위그룹/신규)"
    reason[(reason == "") & (base < min_base)] = f"기준연도 규모 < {min_base:,.0f}"
    reason[reason.index.isin(EXCLUDE_CODES)] = "제외코드"
    return reason[reason == ""].index.tolist(), reason[reason != ""]


# ═════════════════════════════════════════════════════════════
# 2. 지표 계산
# ═════════════════════════════════════════════════════════════
def cagr(a0, a1, years):
    with np.errstate(divide="ignore", invalid="ignore"):
        return (a1 / a0) ** (1 / years) - 1


def max_drawdown(series_df):
    dd = series_df / series_df.cummax() - 1
    return dd.min()


def compute_metrics(monthly, names, classes, start, end):
    mid = end - SHORT_WINDOW
    ann_all = monthly.groupby(monthly.index.year).sum(min_count=1)
    ann = monthly[classes].groupby(monthly.index.year).sum(min_count=12)
    mkt_ann = monthly.drop(columns=[c for c in EXCLUDE_CODES if c in monthly.columns]) \
                     .groupby(monthly.index.year).sum().sum(axis=1)

    yoy = ann.pct_change().loc[start + 1:end]                      # 연도별 성장률
    m = pd.DataFrame(index=classes)
    m["약효분류명"] = names.reindex(classes)
    m[f"{start}"] = ann.loc[start]
    m[f"{mid}"] = ann.loc[mid]
    m[f"{end - 1}"] = ann.loc[end - 1]
    m[f"{end}"] = ann.loc[end]

    m["CAGR_장기"] = cagr(ann.loc[start], ann.loc[end], end - start)
    m["CAGR_단기"] = cagr(ann.loc[mid], ann.loc[end], SHORT_WINDOW)
    m["YoY_최근"] = ann.loc[end] / ann.loc[end - 1] - 1
    m["격차_단기-장기(%p)"] = (m["CAGR_단기"] - m["CAGR_장기"]) * 100
    m["음성장_연수"] = (yoy < 0).sum()
    m["연도별성장률_최저"] = yoy.min()

    ttm = monthly[classes].rolling(12, min_periods=12).sum().dropna(how="all")   # 계절성 제거
    m["MDD_TTM"] = max_drawdown(ttm)
    with np.errstate(divide="ignore", invalid="ignore"):
        m["칼마비율"] = np.where(m["MDD_TTM"] < -0.005, m["CAGR_장기"] / m["MDD_TTM"].abs(), np.nan)

    d_cls = ann.loc[end] - ann.loc[start]
    d_mkt = mkt_ann.loc[end] - mkt_ann.loc[start]
    m["증가액"] = d_cls
    m["시장성장_기여도"] = d_cls / d_mkt
    m["점유율_시작"] = ann.loc[start] / mkt_ann.loc[start]
    m["점유율_최근"] = ann.loc[end] / mkt_ann.loc[end]
    m["점유율변화(%p)"] = (m["점유율_최근"] - m["점유율_시작"]) * 100

    mk = dict(
        long=cagr(mkt_ann.loc[start], mkt_ann.loc[end], end - start),
        short=cagr(mkt_ann.loc[mid], mkt_ann.loc[end], SHORT_WINDOW),
        yoy=mkt_ann.loc[end] / mkt_ann.loc[end - 1] - 1,
        mdd=max_drawdown(monthly.drop(columns=[c for c in EXCLUDE_CODES if c in monthly.columns])
                         .sum(axis=1).rolling(12, min_periods=12).sum().dropna().to_frame())[0],
        ann=mkt_ann,
    )

    # 사분면 · 가속도 라벨
    hi_l, hi_s = m["CAGR_장기"] > mk["long"], m["CAGR_단기"] > mk["short"]
    m["사분면"] = np.select(
        [hi_l & hi_s, ~hi_l & hi_s, hi_l & ~hi_s],
        ["지속성장", "반등·신흥", "둔화"], default="정체·쇠퇴")
    gap = m["CAGR_단기"] - m["CAGR_장기"]
    m["가속도"] = np.select([gap > ACCEL_TOL, gap < -ACCEL_TOL], ["가속", "둔화"], default="유지")

    ttm_out = ttm
    return m.sort_values(f"{end}", ascending=False), yoy, ann, ttm_out, mk



def place_labels(ax, xs, ys, texts, fontsize=9.5, avoid_r=None):
    """겹침을 줄이는 간단한 라벨 배치(그리디). 좌표는 데이터 좌표, 내부 계산은 축 비율."""
    x0, x1 = ax.get_xlim(); y0, y1 = ax.get_ylim()
    fx = lambda v: (v - x0) / (x1 - x0)
    fy = lambda v: (v - y0) / (y1 - y0)
    px = [fx(v) for v in xs]; py = [fy(v) for v in ys]
    ch_w, h = fontsize * 0.00135, fontsize * 0.0032       # 글자폭/높이(축 비율, 대략치)
    placed = []
    cands = [(0.012, 0.012), (0.012, -0.03), (-0.012, 0.012), (-0.012, -0.03),
             (0.012, 0.045), (0.012, -0.06), (-0.012, 0.045), (-0.012, -0.06),
             (0.03, 0.0), (-0.03, 0.0), (0.03, 0.07), (-0.03, -0.07)]
    for i, t in enumerate(texts):
        w = ch_w * len(t) * 1.7
        best, best_pen = None, 1e9
        for dx, dy in cands:
            left = px[i] + dx if dx > 0 else px[i] + dx - w
            rect = (left, py[i] + dy, left + w, py[i] + dy + h)
            pen = 0
            if rect[0] < 0 or rect[2] > 1 or rect[1] < 0 or rect[3] > 1:
                pen += 5
            for r in placed:
                ox = max(0, min(rect[2], r[2]) - max(rect[0], r[0]))
                oy = max(0, min(rect[3], r[3]) - max(rect[1], r[1]))
                pen += 40 * ox * oy / (w * h)
            for j in range(len(px)):
                if abs(px[j] - (rect[0] + rect[2]) / 2) < w / 2 + 0.008 and abs(py[j] - (rect[1] + rect[3]) / 2) < h / 2 + 0.008 and j != i:
                    pen += 0.6
            pen += 0.02 * (abs(dx) + abs(dy))
            if pen < best_pen:
                best, best_pen = (dx, dy, rect), pen
        dx, dy, rect = best
        placed.append(rect)
        ax.annotate(t, (xs[i], ys[i]), xytext=(x0 + (rect[0] if dx > 0 else rect[2]) * (x1 - x0),
                    y0 + (rect[1] + h / 2) * (y1 - y0)), fontsize=fontsize, zorder=6,
                    ha="left" if dx > 0 else "right", va="center",
                    arrowprops=dict(arrowstyle="-", color="#888", lw=.5, shrinkA=0, shrinkB=2),
                    bbox=dict(boxstyle="round,pad=.15", fc="white", ec="none", alpha=.8))

# ═════════════════════════════════════════════════════════════
# 3. 시각화
# ═════════════════════════════════════════════════════════════
def fig1_growth_matrix(m, mk, ctx, path):
    """장기(X) vs 단기(Y) 성장 버블 매트릭스."""
    start, end, div, unit = ctx["start"], ctx["end"], ctx["div"], ctx["unit"]
    x, y = m["CAGR_장기"] * 100, m["CAGR_단기"] * 100
    lo = np.floor(min(x.quantile(.03), y.quantile(.03), mk["long"] * 100, mk["short"] * 100)) - 2
    hi = np.ceil(max(x.quantile(.97), y.quantile(.97), mk["long"] * 100, mk["short"] * 100)) + 2
    xc, yc = x.clip(lo, hi), y.clip(lo, hi)
    clipped = (x != xc) | (y != yc)
    amt = m[f"{end}"]
    s = 25 + 1300 * (amt / amt.max()) ** 0.8
    col = m["가속도"].map({"가속": C_UP, "유지": C_FLAT, "둔화": C_DOWN})

    fig, ax = plt.subplots(figsize=(12, 9))
    ml, ms = mk["long"] * 100, mk["short"] * 100
    ax.axvspan(ml, hi, ymin=(ms - lo) / (hi - lo), ymax=1, color="#e8f1fa", zorder=0)
    ax.axvspan(lo, ml, ymin=(ms - lo) / (hi - lo), ymax=1, color="#eef7ee", zorder=0)
    ax.axvspan(ml, hi, ymin=0, ymax=(ms - lo) / (hi - lo), color="#fdf3e4", zorder=0)
    ax.axvspan(lo, ml, ymin=0, ymax=(ms - lo) / (hi - lo), color="#f8eaea", zorder=0)
    ax.plot([lo, hi], [lo, hi], "--", color="#555", lw=1, zorder=1)
    ax.axvline(ml, color=C_MKT, lw=1.2, zorder=1)
    ax.axhline(ms, color=C_MKT, lw=1.2, zorder=1)

    ax.scatter(xc[~clipped], yc[~clipped], s=s[~clipped], c=col[~clipped], alpha=.72,
               edgecolor="white", linewidth=.8, zorder=3)
    ax.scatter(xc[clipped], yc[clipped], s=s[clipped], facecolor="none",
               edgecolor=col[clipped], linewidth=2, zorder=3)

    kw = dict(fontsize=13, fontweight="bold", alpha=.55, zorder=2)
    ax.text(hi - .5, hi - .5, "지속성장\n(장·단기 모두 시장 초과)", ha="right", va="top", color="#1f5f99", **kw)
    ax.text(lo + .5, hi - .5, "반등·신흥\n(최근 들어 시장 초과)", ha="left", va="top", color="#2e7d32", **kw)
    ax.text(hi - .5, lo + .5, "둔화\n(장기 우수 → 최근 부진)", ha="right", va="bottom", color="#b26a00", **kw)
    ax.text(lo + .5, lo + .5, "정체·쇠퇴", ha="left", va="bottom", color="#a33", **kw)
    ax.text(hi - .3, hi - (hi - lo) * .16, "대각선 위 ▲ 최근 가속", ha="right", fontsize=10, color="#555", rotation=0)
    ax.text(hi - .3, lo + (hi - lo) * .18, "대각선 아래 ▼ 최근 둔화", ha="right", fontsize=10, color="#555")

    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
    top = list(dict.fromkeys(list(amt.nlargest(12).index) + list(m["격차_단기-장기(%p)"].nlargest(3).index)
                             + list(m["격차_단기-장기(%p)"].nsmallest(3).index)))
    place_labels(ax, [xc[c] for c in top], [yc[c] for c in top], [short_name(m.loc[c, "약효분류명"]) for c in top])
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:.0f}%"))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:.0f}%"))
    ax.set_xlabel(f"장기 성장: {start}→{end} CAGR", fontsize=12)
    ax.set_ylabel(f"단기 성장: {end - SHORT_WINDOW}→{end} CAGR", fontsize=12)
    ax.set_title(f"약효분류군 장·단기 성장 매트릭스  (검은선 = 전체 시장 장기 {ml:.1f}% / 단기 {ms:.1f}%)",
                 fontsize=15, fontweight="bold", loc="left", pad=14)

    handles = [Line2D([], [], marker="o", ls="", color=C_UP, ms=10, label="가속 (단기−장기 > +2%p)"),
               Line2D([], [], marker="o", ls="", color=C_FLAT, ms=10, label="유지 (±2%p 이내)"),
               Line2D([], [], marker="o", ls="", color=C_DOWN, ms=10, label="둔화 (< −2%p)"),
               Line2D([], [], marker="o", ls="", mfc="none", mec="#333", ms=10, label="축 범위 초과(가장자리 표시)")]
    for v in [x for x in (500 * div, 2000 * div, 10000 * div) if x < amt.max()]:
        sz = 25 + 1300 * (v / amt.max()) ** 0.8
        handles.append(Line2D([], [], marker="o", ls="", color="#bbb", ms=np.sqrt(sz),
                              label=f"{v / div:,.0f} {unit}"))
    ax.legend(handles=handles, loc="lower center", bbox_to_anchor=(.5, -.2), ncol=4, frameon=False,
              fontsize=10, labelspacing=1.6, handletextpad=1.2)
    fig.text(.01, .005, f"버블 크기 = {end}년 규모  |  색 = 단기−장기 격차(가속/둔화)", fontsize=9, color="#666")
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight");
    fig
    plt.close(fig)


def fig2_heatmap(m, yoy, mk, ctx, path, top_n=28):
    """연도별 성장률 히트맵 — CAGR이 숨기는 하락 연도를 색으로 드러냄."""
    start, end, div, unit = ctx["start"], ctx["end"], ctx["div"], ctx["unit"]
    sel = m.nlargest(top_n, f"{end}").index
    years = list(range(start + 1, end + 1))
    mk_yoy = mk["ann"].pct_change().loc[years]

    rows = [mk_yoy.values.tolist() + [mk["long"], mk["short"]]]
    labels = ["■ 전체 시장"]
    right = [f"{mk['ann'].loc[end] / div:,.0f}"]
    for c in sel:
        rows.append(yoy.loc[years, c].tolist() + [m.loc[c, "CAGR_장기"], m.loc[c, "CAGR_단기"]])
        labels.append(f"{short_name(m.loc[c, '약효분류명'], 14)} ({c})")
        right.append(f"{m.loc[c, f'{end}'] / div:,.0f}")
    M = np.array(rows) * 100
    cols = [f"{y}\nYoY" for y in years] + [f"CAGR\n{start}→{end}", f"CAGR\n{end - SHORT_WINDOW}→{end}"]

    fig, ax = plt.subplots(figsize=(11, 0.36 * len(rows) + 2.4))
    norm = TwoSlopeNorm(vmin=-20, vcenter=0, vmax=20)
    im = ax.imshow(np.clip(M, -20, 20), cmap="RdBu", norm=norm, aspect="auto")
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            v = M[i, j]
            if np.isnan(v):
                continue
            ax.text(j, i, (f"{v:+.1f}" if abs(v) < 1 else f"{v:+.0f}"), ha="center", va="center", fontsize=9,
                    fontweight="bold" if v < 0 else "normal",
                    color="white" if abs(v) > 15 else "#111")
            if v < 0 and j < len(years):        # 하락 연도 테두리
                ax.add_patch(plt.Rectangle((j - .5, i - .5), 1, 1, fill=False, ec="#8b0000", lw=1.4))
    ax.axvline(len(years) - .5, color="white", lw=6)
    ax.axhline(.5, color="white", lw=4)
    ax.set_xticks(range(len(cols))); ax.set_xticklabels(cols, fontsize=10)
    ax.xaxis.tick_top()
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=10)
    ax.get_yticklabels()[0].set_fontweight("bold")
    ax2 = ax.twinx(); ax2.set_ylim(ax.get_ylim()); ax2.set_yticks(range(len(labels)))
    ax2.set_yticklabels(right, fontsize=9, color="#555"); ax2.spines[:].set_visible(False)
    ax2.set_ylabel(f"{end}년 규모 ({unit})", fontsize=9, color="#555")
    for a in (ax, ax2):
        a.tick_params(length=0)
        for sp in a.spines.values():
            sp.set_visible(False)
    cb = fig.colorbar(im, ax=ax, orientation="horizontal", pad=.03, fraction=.025, shrink=.5)
    cb.set_label("◀ 감소 (빨강)      0      증가 (파랑) ▶   (%, ±20% 초과는 색 포화)", fontsize=9)
    ax.set_title(f"연도별 성장률 히트맵 — {end}년 규모 상위 {top_n}개  (빨간 테두리 = 마이너스 성장 연도)",
                 fontsize=14, fontweight="bold", loc="left", pad=40)
    fig.savefig(path, dpi=150, bbox_inches="tight");
    fig
    plt.close(fig)


def fig3_contribution(m, mk, ctx, path, k=10):
    """시장 성장에 대한 기여도(증가액)와 점유율 변화."""
    start, end, div, unit = ctx["start"], ctx["end"], ctx["div"], ctx["unit"]
    d = pd.concat([m.nlargest(k + 1, "증가액"), m.nsmallest(5, "점유율변화(%p)")]).drop_duplicates()
    d = d.sort_values("증가액")
    lab = [f"{short_name(n, 13)} ({c})" for c, n in zip(d.index, d["약효분류명"])]
    colors = [C_UP if v > 0 else C_DOWN for v in d["증가액"]]

    fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 0.42 * len(d) + 2.2), sharey=True,
                                 gridspec_kw=dict(width_ratios=[1.35, 1]))
    yy = np.arange(len(d))
    a1.barh(yy, d["증가액"] / div, color=colors, height=.68)
    for i, (v, c) in enumerate(zip(d["증가액"] / div, d["시장성장_기여도"])):
        a1.text(v + (0.01 if v > 0 else -0.01) * abs(d["증가액"] / div).max(), i,
                f"{v:+,.0f}  ({c * 100:+.0f}%)", va="center", ha="left" if v > 0 else "right", fontsize=9)
    a1.axvline(0, color="#333", lw=.8)
    a1.set_yticks(yy); a1.set_yticklabels(lab, fontsize=10)
    a1.set_xlabel(f"{start}→{end} 증가액 ({unit})  · 괄호 = 전체 시장 증가분에서 차지하는 비중", fontsize=10)
    a1.set_title("① 시장 증가분에 대한 기여 (상위 + 점유율 하락 분류군)", fontsize=12, fontweight="bold", loc="left")
    lim = abs(d["증가액"] / div).max() * 1.35
    a1.set_xlim(-lim if (d["증가액"] < 0).any() else -lim * .05, lim)
    a1.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:,.0f}"))

    a2.barh(yy, d["점유율변화(%p)"], color=[C_UP if v > 0 else C_DOWN for v in d["점유율변화(%p)"]], height=.68)
    for i, (v, s0, s1) in enumerate(zip(d["점유율변화(%p)"], d["점유율_시작"], d["점유율_최근"])):
        a2.text(v + (.02 if v > 0 else -.02), i, f"{v:+.1f}%p  ({s0 * 100:.1f}→{s1 * 100:.1f}%)", va="center",
                ha="left" if v > 0 else "right", fontsize=9)
    a2.axvline(0, color="#333", lw=.8)
    a2.set_xlabel("점유율 변화 (%p)", fontsize=10)
    a2.set_title("② 시장 점유율 변화", fontsize=12, fontweight="bold", loc="left")
    l2 = abs(d["점유율변화(%p)"]).max() * 1.9
    a2.set_xlim(-l2, l2)
    a2.tick_params(axis="y", length=0)
    fig.suptitle(f"성장 기여도 & 점유율 변화 ({start}→{end}) — 증가액은 크지만 점유율이 줄면 시장 평균보다 느리게 큰 것",
                 fontsize=14, fontweight="bold", x=.01, ha="left")
    fig.tight_layout(rect=(0, 0, 1, .95))
    fig.savefig(path, dpi=150, bbox_inches="tight");
    fig
    plt.close(fig)


def _spread(vals, gap):
    """끝단 라벨 겹침 방지 (log 공간에서 최소 간격 확보)."""
    order = np.argsort(vals)
    v = np.array(vals, dtype=float)[order]
    for i in range(1, len(v)):
        if v[i] - v[i - 1] < gap:
            v[i] = v[i - 1] + gap
    out = np.empty_like(v); out[order] = v
    return out

def _spread(vals, gap):
    """끝단 라벨 겹침 방지 (log 공간에서 최소 간격 확보)."""
    order = np.argsort(vals)
    v = np.array(vals, dtype=float)[order]
    for i in range(1, len(v)):
        if v[i] - v[i - 1] < gap:
            v[i] = v[i - 1] + gap
    out = np.empty_like(v); out[order] = v
    return out


def fig4_trend_log(m, ttm, mk, ctx, path, top_n=10):
    """로그 스케일 추이(2020년 연간=100). 기울기 = 성장률."""
    start, end = ctx["start"], ctx["end"]
    sel = m.nlargest(top_n, f"{end}").index
    base = m.loc[sel, f"{start}"]
    idx = ttm.loc[ttm.index >= f"{start}-12-01", sel] / base * 100
    mk_ttm = ctx["mkt_ttm"].loc[idx.index]
    mk_idx = mk_ttm / mk["ann"].loc[start] * 100

    fig, ax = plt.subplots(figsize=(12, 7.5))
    cmap = plt.get_cmap("tab10")
    for i, c in enumerate(sel):
        ax.plot(idx.index, idx[c], color=cmap(i), lw=2, alpha=.9)
    ax.plot(mk_idx.index, mk_idx, color=C_MKT, lw=3, ls="--", label="전체 시장")
    ends = list(idx.iloc[-1].values) + [mk_idx.iloc[-1]]
    pos = 10 ** _spread(np.log10(ends), 0.028)
    ymax = max(pos.max(), max(ends)) * 1.05
    ax.set_yscale("log")
    ax.axhline(100, color="#aaa", lw=.8)
    ticks = [t for t in (80, 90, 100, 110, 125, 150, 175, 200, 250, 300) if t <= ymax]
    ax.set_yticks(ticks); ax.set_yticklabels([str(t) for t in ticks]); ax.minorticks_off()
    ax.grid(True, which="both", axis="y", alpha=.25)

    names_ = [f"{short_name(m.loc[c, '약효분류명'], 12)}  {m.loc[c, 'CAGR_장기'] * 100:+.1f}%" for c in sel] \
             + [f"전체 시장  {mk['long'] * 100:+.1f}%"]
    colors = [cmap(i) for i in range(len(sel))] + [C_MKT]
    x_end = idx.index[-1] + pd.Timedelta(days=25)
    for n, p, e, cl in zip(names_, pos, ends, colors):
        ax.annotate(n, (idx.index[-1], e), xytext=(x_end, p), fontsize=10, color=cl, va="center",
                    fontweight="bold", annotation_clip=False,
                    arrowprops=dict(arrowstyle="-", color=cl, lw=.6, alpha=.6))
    ax.set_xlim(idx.index[0], idx.index[-1] + pd.Timedelta(days=330)); ax.set_ylim(idx.min().min() * .96, ymax)
    dec = [d for d in idx.index if d.month == 12]           # 12개월 누적이 '해당 연도 합계'가 되는 시점
    ax.set_xticks(dec); ax.set_xticklabels([f"{d.year}.12" for d in dec])
    ax.set_xlabel(f"기준월 (각 시점 = 직전 12개월 누적, 데이터 끝: {idx.index[-1].year}.{idx.index[-1].month:02d})", fontsize=10)
    ax.set_ylabel(f"12개월 누적(TTM) 지수  ({start}년 = 100, 로그축)", fontsize=11)
    ax.set_title(f"규모 상위 {top_n}개 분류군 추이 — 로그 스케일에서 기울기가 같으면 성장률이 같음",
                 fontsize=14, fontweight="bold", loc="left")
    fig.text(.01, -.01, "TTM(최근 12개월 합계)을 쓰면 월별 계절성이 제거됩니다. 라벨 % = 장기 CAGR.", fontsize=9, color="#666")
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight"); plt.close(fig)


def fig5_risk_return(m, mk, ctx, path):
    """성장(CAGR) vs 최대낙폭(MDD): 우상단 = 고성장·저낙폭."""
    start, end, div, unit = ctx["start"], ctx["end"], ctx["div"], ctx["unit"]
    x, y = m["MDD_TTM"] * 100, m["CAGR_장기"] * 100
    ylo = max(y.quantile(.03), -25) - 2; yhi = min(y.quantile(.97), 40) + 2
    xlo = max(x.quantile(.05), -40) - 2
    yc = y.clip(ylo, yhi); xc = x.clip(xlo, 2)
    amt = m[f"{end}"]
    s = 25 + 1200 * (amt / amt.max()) ** 0.8
    cmap = {0: "#2b7bba", 1: "#f0a30a", 2: C_DOWN}
    col = m["음성장_연수"].clip(upper=2).map(cmap)

    fig, ax = plt.subplots(figsize=(12, 8.5))
    ax.axhline(mk["long"] * 100, color=C_MKT, lw=1.2)
    ax.axvline(mk["mdd"] * 100, color=C_MKT, lw=1.2)
    ax.text(0, ylo, "", )
    ax.scatter(xc, yc, s=s, c=col, alpha=.72, edgecolor="white", linewidth=.8, zorder=3)
    ax.set_xlim(xlo, 2); ax.set_ylim(ylo, yhi)
    xl = ax.get_xlim()
    top = list(dict.fromkeys(list(amt.nlargest(12).index) + list(m["칼마비율"].nlargest(2).index)
                             + list(m["MDD_TTM"].nsmallest(3).index)))
    place_labels(ax, [xc[c] for c in top], [yc[c] for c in top], [short_name(m.loc[c, "약효분류명"]) for c in top])
    kw = dict(fontsize=13, fontweight="bold", alpha=.5)
    ax.text(1.5, yhi - .5, "고성장 · 저낙폭\n(가장 안정적)", ha="right", va="top", color="#1f5f99", **kw)
    ax.text(xl[0] + .5, yhi - .5, "고성장 · 고변동", ha="left", va="top", color="#b26a00", **kw)
    ax.text(1.5, ylo + .5, "저성장 · 저낙폭", ha="right", va="bottom", color="#777", **kw)
    ax.text(xl[0] + .5, ylo + .5, "저성장 · 고낙폭\n(위험)", ha="left", va="bottom", color="#a33", **kw)
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:.0f}%"))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:.0f}%"))
    ax.set_xlabel("최대낙폭 MDD (12개월 누적 기준, 0에 가까울수록 안정)", fontsize=12)
    ax.set_ylabel(f"장기 CAGR ({start}→{end})", fontsize=12)
    ax.set_title(f"성장 vs 하락 위험  (검은선 = 전체 시장: CAGR {mk['long'] * 100:.1f}%, MDD {mk['mdd'] * 100:.1f}%)",
                 fontsize=15, fontweight="bold", loc="left", pad=12)
    handles = [Line2D([], [], marker="o", ls="", color=cmap[0], ms=10, label="마이너스 성장 연도 0회"),
               Line2D([], [], marker="o", ls="", color=cmap[1], ms=10, label="1회"),
               Line2D([], [], marker="o", ls="", color=cmap[2], ms=10, label="2회 이상")]
    ax.legend(handles=handles, loc="lower center", bbox_to_anchor=(.5, -.14), ncol=3, frameon=False, fontsize=10)
    fig.text(.01, .005, "버블 크기 = 최근 연도 규모  |  CAGR이 못 보는 '가는 길의 험함'을 MDD·색으로 보완", fontsize=9, color="#666")
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight");
    fig
    plt.close(fig)


# ═════════════════════════════════════════════════════════════
# 4. 엑셀/CSV 저장
# ═════════════════════════════════════════════════════════════
def save_tables(m, yoy, ann, excluded, ctx, outdir):
    from openpyxl.formatting.rule import ColorScaleRule
    from openpyxl.utils import get_column_letter

    start, end, div = ctx["start"], ctx["end"], ctx["div"]
    tbl = m.copy()
    tbl.index.name = "약효분류코드"
    for c in tbl.columns:                         # 금액 컬럼 → 억원 환산
        if c in [str(y) for y in range(start, end + 1)] or c == "증가액":
            tbl[c] = tbl[c] / div
    tbl.to_csv(os.path.join(outdir, "growth_metrics.csv"), encoding="utf-8-sig", float_format="%.4f")

    pct_cols = ["CAGR_장기", "CAGR_단기", "YoY_최근", "연도별성장률_최저", "MDD_TTM", "시장성장_기여도",
                "점유율_시작", "점유율_최근"]
    path = os.path.join(outdir, "growth_metrics.xlsx")
    with pd.ExcelWriter(path, engine="openpyxl") as w:
        tbl.to_excel(w, sheet_name="지표")
        (yoy.rename(columns=lambda c: f"{c} {m.loc[c, '약효분류명']}" if c in m.index else c).T
         ).to_excel(w, sheet_name="연도별성장률")
        (ann.div(div)).T.assign(약효분류명=lambda d: d.index.map(m["약효분류명"])).to_excel(w, sheet_name="연도별규모")
        excluded.rename("제외사유").to_frame().to_excel(w, sheet_name="제외분류")
        ws = w.sheets["지표"]
        ws.freeze_panes = "C2"
        headers = [c.value for c in ws[1]]
        for j, h in enumerate(headers, 1):
            L = get_column_letter(j)
            ws.column_dimensions[L].width = 22 if h == "약효분류명" else 13
            for cell in ws[L][1:]:
                if h in pct_cols:
                    cell.number_format = "0.0%"
                elif h in [str(y) for y in range(start, end + 1)] or h == "증가액":
                    cell.number_format = "#,##0"
                elif h in ("격차_단기-장기(%p)", "점유율변화(%p)", "칼마비율"):
                    cell.number_format = "0.0"
            if h in ("CAGR_장기", "CAGR_단기", "YoY_최근", "격차_단기-장기(%p)", "점유율변화(%p)"):
                ws.conditional_formatting.add(
                    f"{L}2:{L}{ws.max_row}",
                    ColorScaleRule(start_type="num", start_value=-0.2 if "%p" not in h else -10, start_color="F8696B",
                                   mid_type="num", mid_value=0, mid_color="FFFFFF",
                                   end_type="num", end_value=0.2 if "%p" not in h else 10, end_color="5A8AC6"))
        w.sheets["연도별성장률"].freeze_panes = "B2"
        for L in ("A", "B"):
            w.sheets["연도별성장률"].column_dimensions[L].width = 30
        for row in w.sheets["연도별성장률"].iter_rows(min_row=2, min_col=2):
            for cell in row:
                cell.number_format = "0.0%"


# ═════════════════════════════════════════════════════════════
# 5. main
# ═════════════════════════════════════════════════════════════
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--csv", default=CSV_PATH)
    ap.add_argument("--out", default="output")
    ap.add_argument("--metric", default="사용금액_백만원", choices=list(UNITS))
    ap.add_argument("--start", type=int, default=2020)
    ap.add_argument("--end", type=int, default=2025)
    ap.add_argument("--min-base", type=float, default=MIN_BASE,
                    help="기준연도 최소 규모(지표 원단위). 작은 분류군의 성장률 과장 방지. 기본 100(백만원=1억)")
    # a = ap.parse_args() # not working at colab
    a, unknown = ap.parse_known_args()

    setup_font()
    os.makedirs(a.out, exist_ok=True)
    div, unit = UNITS[a.metric]
    ctx = dict(start=a.start, end=a.end, div=div, unit=unit)

    monthly, names = load(a.csv, a.metric)
    classes, excluded = select_classes(monthly, a.start, a.min_base)
    m, yoy, ann, ttm, mk = compute_metrics(monthly, names, classes, a.start, a.end)
    ctx["mkt_ttm"] = monthly.drop(columns=[c for c in EXCLUDE_CODES if c in monthly.columns]) \
                            .sum(axis=1).rolling(12, min_periods=12).sum().dropna()

    save_tables(m, yoy, ann, excluded, ctx, a.out)
    fig1_growth_matrix(m, mk, ctx, os.path.join(a.out, "1_growth_matrix.png"))
    fig2_heatmap(m, yoy, mk, ctx, os.path.join(a.out, "2_yearly_heatmap.png"))
    fig3_contribution(m, mk, ctx, os.path.join(a.out, "3_contribution.png"))
    fig4_trend_log(m, ttm, mk, ctx, os.path.join(a.out, "4_trend_log.png"))
    fig5_risk_return(m, mk, ctx, os.path.join(a.out, "5_risk_return.png"))

    # 콘솔 요약
    print(f"분석 대상 {len(classes)}개 분류군 (제외 {len(excluded)}개)")
    print(f"전체 시장  장기CAGR {mk['long']:.1%} | 단기CAGR {mk['short']:.1%} | 최근YoY {mk['yoy']:.1%} | MDD {mk['mdd']:.1%}")
    cols = ["약효분류명", "CAGR_장기", "CAGR_단기", "YoY_최근", "격차_단기-장기(%p)"]
    big = m.nlargest(40, f"{a.end}")
    print("\n[규모 상위 40 중 최근 가속 TOP5]\n", big.nlargest(5, "격차_단기-장기(%p)")[cols].round(3))
    print("\n[규모 상위 40 중 최근 둔화 TOP5]\n", big.nsmallest(5, "격차_단기-장기(%p)")[cols].round(3))
    print(f"\n저장 완료 → {os.path.abspath(a.out)}")


if __name__ == "__main__":
    main()

findfont: Failed to find font weight bold for NanumGothic, now using 400.
findfont: Failed to find font weight bold for NanumGothic, now using 400.
findfont: Failed to find font weight bold for NanumGothic, now using 400.
findfont: Failed to find font weight bold for NanumGothic, now using 400.
findfont: Failed to find font weight bold for NanumGothic, now using 400.
findfont: Failed to find font weight bold for NanumGothic, now using 400.


분석 대상 114개 분류군 (제외 14개)
전체 시장  장기CAGR 7.7% | 단기CAGR 6.3% | 최근YoY 7.0% | MDD -0.3%

[규모 상위 40 중 최근 가속 TOP5]
                    약효분류명  CAGR_장기  CAGR_단기  YoY_최근  격차_단기-장기(%p)
629            기타의 화학요법제    0.149    0.372   0.876        22.340
249  기타의 호르몬제(항호르몬제를 포함)    0.095    0.165   0.156         7.026
235              최토제,진토제    0.103    0.162   0.186         5.926
634                혈액제제류    0.066    0.102   0.168         3.600
217                혈관확장제   -0.023    0.002  -0.004         2.529

[규모 상위 40 중 최근 둔화 TOP5]
                          약효분류명  CAGR_장기  CAGR_단기  YoY_최근  격차_단기-장기(%p)
619  기타의 항생물질 제제(복합항생물질제제를 포함)    0.117   -0.056  -0.242       -17.264
222                      진해거담제    0.135   -0.036  -0.129       -17.113
141                     항히스타민제    0.077   -0.020  -0.026        -9.737
229                 기타의 호흡기관용약    0.156    0.066   0.055        -9.014
241                   뇌하수체호르몬제    0.175    0.091   0.069        -8.457

저장 완료 → /workspaces/healthcareDA/src/output


In [ ]:
# sudo apt-get install -y fonts-nanum 후 커널 재시작

# !sudo apt-get update
# !sudo apt-get install -y fonts-nanum

# sudo fc-cache -fv              # refresh the system font cache
# rm -rf ~/.cache/matplotlib     # delete matplotlib's stale font list

# import matplotlib.font_manager as fm

# names = {f.name for f in fm.fontManager.ttflist}
# print(sorted(n for n in names if "Nanum" in n or "Gothic" in n))

from pathlib import Path
import urllib.request
import matplotlib.font_manager as fm

font_dir = Path.home() / ".fonts"
font_dir.mkdir(exist_ok=True)
font_path = font_dir / "NanumGothic.ttf"

if not font_path.exists():
    url = "https://github.com/google/fonts/raw/main/ofl/nanumgothic/NanumGothic-Regular.ttf"
    urllib.request.urlretrieve(url, font_path)

fm.fontManager.addfont(str(font_path))

# !pip install openpyxl
# !pip install koreanize-matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]1/2 [openpyxl]


In [ ]:
# ZERO_FLOOR = 0.1           # 최종 분석대상의 초기 0값(censored)을 치환할 값 (지표 원단위, 금액=백만원)
# ZERO_LEADING_ONLY = True   # True: 첫 양수값 이전의 초기 0만 치환 / False: 모든 0 치환
#  결과 똑같이나옴.. 애초에 다 걸러냄

## TEST

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests


# ============================================================
# 0. 설정
# ============================================================

INPUT_FILE = RAW_PATH

TARGET_CODES = [
    142, 215, 216, 218, 219, 235, 238, 249, 322, 325,
    394, 421, 429, 612, 629, 632, 634, 728
]

FDR_ALPHA = 0.05

# Residual volatility:
# "과도한 변동성"을 상위 25%로 정의
RESIDUAL_VOLATILITY_QUANTILE = 0.75

# HAC
HAC_MAXLAGS = 12

# CI
CI_ALPHA = 0.05


# ============================================================
# 1. 데이터 로드
# ============================================================

df = pd.read_csv(INPUT_FILE)

# Ensure '약효분류코드' is numeric for filtering
df["약효분류코드"] = pd.to_numeric(df["약효분류코드"], errors="coerce").astype("Int64")
df["약효분류명"] = df["약효분류명"].astype("string")
df["진료년월"] = pd.to_datetime(df["진료년월"], format="%Y-%m", errors="coerce")
df["사용금액_백만원"] = pd.to_numeric(df["사용금액_백만원"], errors="coerce")
df["수량_천개"] = pd.to_numeric(df["수량_천개"], errors="coerce")

df = df[
    df["약효분류코드"].isin(TARGET_CODES)
].copy()

print(f"Shape of df after filtering: {df.shape}")
if df.empty:
    print("Warning: df is empty after filtering by TARGET_CODES. This might lead to an empty res_df and subsequent errors.")
    print("Please check if '약효분류코드' in your CSV data contains any values from TARGET_CODES.")


# ============================================================
# 2. 날짜 및 기본 전처리
# ============================================================

df["date"] = pd.to_datetime(
    df["진료년월"].astype(str),
    format="%Y%m",
    errors="coerce"
)

df["month_num"] = df["date"].dt.month

df["사용금액_백만원"] = pd.to_numeric(
    df["사용금액_백만원"],
    errors="coerce"
)

df = df.dropna(
    subset=["date"]
).copy()

df = df.sort_values(
    ["약효분류코드", "date"]
)

# -*- coding: utf-8 -*-
"""
약효분류군 후보 선별 파이프라인
  Month-FE + WLS + HAC(Newey-West) -> Trend β / t / p / CI / FDR(BH) / YoY median / Residual
  Funnel: 최근36M -> Trend β -> YoY median -> FDR -> Residual volatility -> 최근24M momentum

[제외 규칙 원칙]
  * 부호 기반 제외(β≤0, YoY≤0, 모멘텀 훼손)는 '통계적으로 유의(p<ALPHA)'할 때만 제외한다.
    유의하지 않으면 제외 로직을 skip(=통과 처리)하고 skipped_rules 에 기록한다.
  * FDR 단계는 그 자체가 유의성 검정이므로 skip 없이 항상 적용한다.
"""
import os
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager

import warnings
%matplotlib inline
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from matplotlib import font_manager as fm
from matplotlib.colors import TwoSlopeNorm
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter, ScalarFormatter

warnings.filterwarnings("ignore")


def setup_font():
    try:
        import koreanize_matplotlib  # noqa: F401
    except ImportError:
        pass
    names = {f.name for f in fm.fontManager.ttflist}
    fam = next((f for f in ['NanumGothic', 'Malgun Gothic', 'AppleGothic',
                            'NanumBarunGothic', 'Noto Sans CJK KR', 'Noto Sans CJK JP'] if f in names), None)
    if fam is None:
        raise RuntimeError("한글 폰트 없음: sudo apt-get install -y fonts-nanum 후 커널 재시작")
    # sns.set_theme(style='whitegrid', font=fam, rc={'axes.unicode_minus': False})

    plt.rcParams["axes.unicode_minus"] = False
    plt.rcParams["figure.dpi"] = 100
    plt.rcParams["axes.spines.top"] = False
    plt.rcParams["axes.spines.right"] = False
    plt.rcParams['font.family'] = fam
    return fam
setup_font()
# ------------------------------------------------------------------ 설정
# DATA_PATH = "/mnt/user-data/uploads/meft_2020-2025.csv"
OUT_DIR = "outputs"
VALUE_COL = "사용금액_백만원"          # 분석 대상 지표 (수량_천개 로도 변경 가능)
TARGET_CODES = [142, 215, 216, 218, 219, 235, 238, 249, 322, 325,
                394, 421, 429, 612, 629, 632, 634, 728]

ALPHA = 0.05                 # 유의수준 (CI = 1-ALPHA)
FDR_Q = 0.05                 # BH-FDR 기준
RECENT_LONG = 36             # Step1 윈도우(개월)
RECENT_MOM = 24              # Step6 모멘텀 윈도우(개월)
WEIGHT_MODE = "level"        # 'level': 금액 비례 가중 / 'equal': OLS
VOL_Z_CUT = 1.645            # 잔차변동성 robust-z 컷 (단측 5%)
BOOT_B, BOOT_BLOCK, SEED = 5000, 6, 42
SKIP_IF_NOT_SIGNIFICANT = True

STAGES = ["S1_최근36M_추세양수", "S2_Trend_β>0", "S3_YoY_median>0",
          "S4_FDR<0.05", "S5_잔차변동성", "S6_최근24M_모멘텀"]
STAGE_LABEL = {
    "S1_최근36M_추세양수": "최근 36개월 양수 미충족",
    "S2_Trend_β>0": "Trend β ≤ 0",
    "S3_YoY_median>0": "YoY median ≤ 0",
    "S4_FDR<0.05": "FDR ≥ 0.05",
    "S5_잔차변동성": "과도한 residual volatility",
    "S6_최근24M_모멘텀": "최근 2년 momentum 훼손",
}


# ------------------------------------------------------------------ 통계 함수
def nw_lag(n):
    """Newey-West 자동 lag: floor(4*(n/100)^(2/9))"""
    return max(1, int(np.floor(4 * (n / 100) ** (2 / 9))))


def wls_hac(y, X, w, lag):
    """WLS 추정 + Newey-West(Bartlett) HAC 공분산 (small-sample n/(n-k) 보정)."""
    n, k = X.shape
    sw = np.sqrt(w)
    Xw, yw = X * sw[:, None], y * sw
    bread = np.linalg.inv(Xw.T @ Xw)
    beta = bread @ Xw.T @ yw
    resid = y - X @ beta
    u = Xw * (sw * resid)[:, None]              # score_i = x_i * w_i * e_i
    S = u.T @ u
    for l in range(1, lag + 1):
        G = u[l:].T @ u[:-l]
        S += (1 - l / (lag + 1)) * (G + G.T)
    V = bread @ S @ bread * n / (n - k)
    return beta, V, resid, n - k


def coef_summary(b, se, df):
    t = b / se
    p = 2 * stats.t.sf(abs(t), df)
    tc = stats.t.ppf(1 - ALPHA / 2, df)
    return t, p, b - tc * se, b + tc * se


def month_dummies(months):
    return np.column_stack([(months == m).astype(float) for m in range(2, 13)])


def fit_full(y, months, w):
    """log(y) ~ 1 + trend(년 단위) + Month-FE(11) : WLS + HAC"""
    n = len(y)
    t = np.arange(n) / 12.0
    X = np.column_stack([np.ones(n), t, month_dummies(months)])
    beta, V, resid, df = wls_hac(y, X, w, nw_lag(n))
    fe = np.r_[0.0, beta[2:]]                    # month 1 = 0 기준
    return beta, V, resid, df, fe


def fit_window(y_sa, w):
    """계절조정된 y 에 대한 [1, trend] WLS+HAC (윈도우 추세)"""
    n = len(y_sa)
    t = np.arange(n) / 12.0
    X = np.column_stack([np.ones(n), t])
    beta, V, resid, df = wls_hac(y_sa, X, w, nw_lag(n))
    return beta[1], np.sqrt(V[1, 1]), df, nw_lag(n)


def block_bootstrap_median(x, B, block, rng):
    n = len(x)
    nb = int(np.ceil(n / block))
    starts = rng.integers(0, n - block + 1, size=(B, nb))
    idx = (starts[:, :, None] + np.arange(block)).reshape(B, -1)[:, :n]
    return np.median(x[idx], axis=1)


def bh_fdr(p):
    p = np.asarray(p)
    m = len(p)
    order = np.argsort(p)
    q = np.empty(m)
    ranked = p[order] * m / (np.arange(m) + 1)
    q[order] = np.minimum.accumulate(ranked[::-1])[::-1].clip(max=1)
    return q


def pct(b):                                       # 연 로그변화 -> 연 %성장
    return (np.exp(b) - 1) * 100


# ------------------------------------------------------------------ 1) 통계치 계산
def compute_stats(df):
    rng = np.random.default_rng(SEED)
    rows = []
    for code in TARGET_CODES:
        d = df[df["약효분류코드"] == code].sort_values("진료년월")
        name = d["약효분류명"].iloc[0]
        raw = d[VALUE_COL].to_numpy(float)
        months = d["월"].to_numpy(int)
        n = len(d)
        rec = dict(code=code, name=name, n_obs=n,
                   n_missing=int(np.isnan(raw).sum()), n_nonpositive=int((raw <= 0).sum()))
        if n < 60 or rec["n_missing"] > 0 or rec["n_nonpositive"] > 0:
            rec["data_flag"] = "데이터 결측/비양수 존재 - 확인 필요"
        y = np.log(raw)
        w = raw / raw.mean() if WEIGHT_MODE == "level" else np.ones(n)

        # ---- 전체기간 Month-FE + WLS + HAC
        beta, V, resid, df_r, fe = fit_full(y, months, w)
        b, se = beta[1], np.sqrt(V[1, 1])
        t, p, lo, hi = coef_summary(b, se, df_r)
        rec.update(trend_beta=b, trend_se=se, trend_t=t, trend_p=p,
                   trend_ci_lo=lo, trend_ci_hi=hi,
                   trend_pct_yr=pct(b), trend_pct_ci_lo=pct(lo), trend_pct_ci_hi=pct(hi),
                   hac_lag=nw_lag(n), df_resid=df_r)

        # ---- 잔차 변동성 (log 스케일 residual sd) + χ² CI
        sd = np.sqrt(np.sum(resid ** 2) / df_r)
        rec.update(resid_sd=sd,
                   resid_sd_ci_lo=sd * np.sqrt(df_r / stats.chi2.ppf(1 - ALPHA / 2, df_r)),
                   resid_sd_ci_hi=sd * np.sqrt(df_r / stats.chi2.ppf(ALPHA / 2, df_r)))

        # ---- 계절조정 시리즈 (전체기간 Month-FE 제거)
        y_sa = y - fe[months - 1]

        # ---- 최근 36M / 24M 윈도우 추세 (계절조정 후)
        for tag, win in (("r36", RECENT_LONG), ("r24", RECENT_MOM)):
            bw, sew, dfw, lg = fit_window(y_sa[-win:], w[-win:])
            tw, pw, lw, hw = coef_summary(bw, sew, dfw)
            rec.update({f"{tag}_beta": bw, f"{tag}_se": sew, f"{tag}_t": tw, f"{tag}_p": pw,
                        f"{tag}_ci_lo": lw, f"{tag}_ci_hi": hw,
                        f"{tag}_pct_yr": pct(bw), f"{tag}_pct_ci_lo": pct(lw),
                        f"{tag}_pct_ci_hi": pct(hw), f"{tag}_hac_lag": lg})

        # ---- YoY median (%) : moving-block bootstrap CI / p (H0: median ≤ 0)
        yoy = (raw[12:] / raw[:-12] - 1) * 100
        med = np.median(yoy)
        bm = block_bootstrap_median(yoy, BOOT_B, BOOT_BLOCK, rng)
        rec.update(yoy_median=med, yoy_n=len(yoy),
                   yoy_ci_lo=np.quantile(bm, ALPHA / 2), yoy_ci_hi=np.quantile(bm, 1 - ALPHA / 2),
                   yoy_p=(np.sum(bm <= 0) + 1) / (BOOT_B + 1),      # 단측(성장>0) p
                   yoy_pos_share=np.mean(yoy > 0))
        rows.append(rec)

    res = pd.DataFrame(rows)
    res["trend_fdr_q"] = bh_fdr(res["trend_p"].to_numpy())          # 18개 전체 family
    # 잔차변동성: log(sd) robust z (median/MAD), 단측 p
    ls = np.log(res["resid_sd"])
    mad = stats.median_abs_deviation(ls, scale="normal")
    res["resid_sd_robust_z"] = (ls - ls.median()) / mad
    res["resid_sd_p"] = stats.norm.sf(res["resid_sd_robust_z"])
    return res


# ------------------------------------------------------------------ 2) Funnel
def run_funnel(res):
    res = res.copy()
    res["status"], res["drop_stage"], res["drop_reason"] = "PASS", "", ""
    res["skipped_rules"] = ""
    log = []

    def sign_rule(r, stage, cond_bad, p, desc_bad, desc_skip):
        """부호 기반 규칙: 나쁜 조건 & 유의 → 탈락 / 나쁜 조건 & 비유의 → skip"""
        if not cond_bad:
            return None
        if (not SKIP_IF_NOT_SIGNIFICANT) or p < ALPHA:
            return desc_bad
        r["skipped_rules"] += f"[{stage}] {desc_skip}; "
        return None

    for stage in STAGES:
        alive = res["status"] == "PASS"
        n_in = int(alive.sum())
        dropped = []
        for i in res.index[alive]:
            r = res.loc[i]
            why = None
            if stage == STAGES[0]:
                why = sign_rule(r, stage, r.r36_beta <= 0, r.r36_p,
                                f"최근36M β={r.r36_beta:.3f}(연 {r.r36_pct_yr:.1f}%) ≤0, p={r.r36_p:.3f}",
                                f"최근36M β≤0 이나 비유의(p={r.r36_p:.3f})")
            elif stage == STAGES[1]:
                why = sign_rule(r, stage, r.trend_beta <= 0, r.trend_p,
                                f"Trend β={r.trend_beta:.3f} ≤0, p={r.trend_p:.3f}",
                                f"Trend β≤0 이나 비유의(p={r.trend_p:.3f})")
            elif stage == STAGES[2]:
                why = sign_rule(r, stage, r.yoy_median <= 0, 1 - r.yoy_p,  # 하락 방향 p
                                f"YoY median={r.yoy_median:.2f}% ≤0 (CI {r.yoy_ci_lo:.1f}~{r.yoy_ci_hi:.1f})",
                                f"YoY median≤0 이나 CI가 0 포함")
            elif stage == STAGES[3]:
                if r.trend_fdr_q >= FDR_Q:
                    why = f"FDR q={r.trend_fdr_q:.3f} ≥ {FDR_Q} (raw p={r.trend_p:.3f})"
            elif stage == STAGES[4]:
                if r.resid_sd_robust_z > VOL_Z_CUT and r.resid_sd_p < ALPHA:
                    why = (f"residual sd={r.resid_sd:.3f} (robust z={r.resid_sd_robust_z:.2f}, "
                           f"p={r.resid_sd_p:.3f}) 과대")
                elif r.resid_sd_robust_z > VOL_Z_CUT:
                    r["skipped_rules"] += f"[{stage}] 변동성 과대이나 비유의; "
            elif stage == STAGES[5]:
                why = sign_rule(r, stage, r.r24_beta < 0, r.r24_p,
                                f"최근24M β={r.r24_beta:.3f}(연 {r.r24_pct_yr:.1f}%) <0, p={r.r24_p:.3f}",
                                f"최근24M β<0 이나 비유의(p={r.r24_p:.3f})")
            res.loc[i, "skipped_rules"] = r["skipped_rules"]
            if why:
                res.loc[i, ["status", "drop_stage", "drop_reason"]] = ["DROP", stage, why]
                dropped.append(i)
        log.append(dict(stage=stage, label=STAGE_LABEL[stage], n_in=n_in,
                        n_dropped=len(dropped), n_pass=n_in - len(dropped),
                        dropped_codes=",".join(res.loc[dropped, "code"].astype(str))))
    res.loc[res["status"] == "PASS", "status"] = "FINAL"
    return res, pd.DataFrame(log)


# ------------------------------------------------------------------ 3) 시각화
def set_font():
    for f in ["Malgun Gothic", "NanumGothic", "Noto Sans CJK KR", "Noto Sans CJK JP", "AppleGothic"]:
        if any(f == x.name for x in font_manager.fontManager.ttflist):
            plt.rcParams["font.family"] = f
            break
    plt.rcParams["axes.unicode_minus"] = False


def chart_funnel(flog, n0, path):
    labels = ["대상 약효분류군"] + [STAGE_LABEL[s].replace(" 미충족", "").replace(" ≤ 0", " > 0") for s in flog.stage]
    labels = ["18개 후보군"] + ["통과: " + l for l in labels[1:]]
    vals = [n0] + list(flog["n_pass"])
    fig, ax = plt.subplots(figsize=(10, 5.5))
    cols = plt.cm.Blues(np.linspace(0.35, 0.9, len(vals)))
    cols[-1] = (0.18, 0.62, 0.35, 1)
    y = np.arange(len(vals))[::-1]
    ax.barh(y, vals, color=cols, height=0.7)
    for yi, v, i in zip(y, vals, range(len(vals))):
        drop = "" if i == 0 else f"   (−{vals[i-1]-v})"
        ax.text(v + 0.2, yi, f"{v}{drop}", va="center", fontsize=10)
    ax.set_yticks(y); ax.set_yticklabels(labels)
    ax.set_xlim(0, n0 * 1.18); ax.set_xlabel("남은 후보 수")
    ax.set_title("Chart 1. Candidate Funnel", fontsize=13, weight="bold")
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout(); fig.savefig(path, dpi=160); plt.close(fig)


def chart_drop(res, path):
    cnt = res[res.status == "DROP"]["drop_stage"].value_counts().reindex(STAGES, fill_value=0)
    names = [STAGE_LABEL[s] for s in cnt.index] + ["최종 후보"]
    vals = list(cnt.values) + [int((res.status == "FINAL").sum())]
    fig, ax = plt.subplots(figsize=(10, 5))
    cols = ["#d95f5f"] * len(cnt) + ["#2e9e5b"]
    ax.barh(names[::-1], vals[::-1], color=cols[::-1])
    for i, v in enumerate(vals[::-1]):
        ax.text(v + 0.05, i, str(v), va="center")
    ax.set_xlabel("후보 수"); ax.set_title("Chart 2. Drop Reason (drop_stage별 탈락 개수)", fontsize=13, weight="bold")
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout(); fig.savefig(path, dpi=160); plt.close(fig)


# def chart_scatter(res, path):
#     fig, ax = plt.subplots(figsize=(11, 7))
#     cmap = dict(zip(STAGES, ["#e6550d", "#8c564b", "#9467bd", "#e377c2", "#7f7f7f", "#17becf"]))
#     for _, r in res.iterrows():
#         fin = r.status == "FINAL"
#         c = "#2e9e5b" if fin else cmap[r.drop_stage]
#         ax.errorbar(r.trend_pct_yr, r.resid_sd,
#                     xerr=[[r.trend_pct_yr - r.trend_pct_ci_lo], [r.trend_pct_ci_hi - r.trend_pct_yr]],
#                     fmt="none", ecolor=c, alpha=0.25, lw=1)
#         ax.scatter(r.trend_pct_yr, r.resid_sd, s=96 if fin else 60, c=c,
#                    marker="o" if fin else "X", edgecolor="k", lw=0.6, zorder=3)
#         ax.annotate(str(r.code), (r.trend_pct_yr, r.resid_sd), xytext=(5, 5),
#                     textcoords="offset points", fontsize=9, weight="bold" if fin else None)
#     ax.axvline(0, color="gray", ls="--", lw=1)
#     ax.axhline(np.exp(np.log(res.resid_sd).median() + VOL_Z_CUT *
#                       stats.median_abs_deviation(np.log(res.resid_sd), scale="normal")),
#                color="#7f7f7f", ls=":", lw=1.2)
#     handles = [plt.Line2D([], [], marker="o", ls="", mfc="#2e9e5b", mec="k", label="최종 후보")]
#     handles += [plt.Line2D([], [], marker="X", ls="", mfc=cmap[s], mec="k", label=f"탈락: {STAGE_LABEL[s]}")
#                 for s in STAGES if (res.drop_stage == s).any()]
#     ax.legend(handles=handles, loc="best", fontsize=9)

#     # ax.set_yscale('log')
#     ax.set_xlabel("Trend β (연 성장률 %, Month-FE 제거, 95% CI)")
#     ax.set_ylabel("Residual volatility (log 잔차 SD)")
#     ax.set_title("Candidate Diagnostic Scatter (점선: 변동성 컷 robust z=1.645)",
#                  fontsize=13, weight="bold")
#     ax.grid(alpha=0.25)
#     plt.tight_layout(); fig.savefig(path, dpi=160); plt.close(fig)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset

# adjustText 패키지가 설치되어 있는 경우 텍스트 자동 겹침 방지 적용
try:
    from adjustText import adjust_text
    HAS_ADJUST_TEXT = True
except ImportError:
    HAS_ADJUST_TEXT = False


def chart_scatter(res, path):
    fig, ax = plt.subplots(figsize=(12, 8))
    cmap = dict(zip(STAGES, ["#e6550d", "#8c564b", "#9467bd", "#e377c2", "#7f7f7f", "#17becf"]))

    # ---------------------------------------------------------
    # 1. 메인 플롯
    # ---------------------------------------------------------
    for _, r in res.iterrows():
        fin = r.status == "FINAL"
        c = "#2e9e5b" if fin else cmap[r.drop_stage]

        # 에러바 및 마커
        ax.errorbar(r.trend_pct_yr, r.resid_sd,
                    xerr=[[r.trend_pct_yr - r.trend_pct_ci_lo], [r.trend_pct_ci_hi - r.trend_pct_yr]],
                    fmt="none", ecolor=c, alpha=0.35, lw=1.5)
        ax.scatter(r.trend_pct_yr, r.resid_sd, s=120 if fin else 80, c=c,
                   marker="o" if fin else "X", edgecolor="k", lw=0.7, zorder=3)

        # 이상치 영역(Vol > 0.25 또는 Trend > 40%)만 메인 플롯에 라벨 표시
        if r.resid_sd > 0.25 or r.trend_pct_yr > 40:
            ax.annotate(str(r.code), (r.trend_pct_yr, r.resid_sd), xytext=(6, 6),
                        textcoords="offset points", fontsize=10, weight="bold" if fin else None)

    # 수직선(0%) 및 변동성 컷 기준선
    ax.axvline(0, color="gray", ls="--", lw=1.2)
    vol_cut = np.exp(np.log(res.resid_sd).median() + VOL_Z_CUT *
                     stats.median_abs_deviation(np.log(res.resid_sd), scale="normal"))
    ax.axhline(vol_cut, color="#7f7f7f", ls=":", lw=1.5)

    # 범례 설정 (우측 상단 서브플롯과의 중첩 방지를 위해 좌상단 배치)
    handles = [plt.Line2D([], [], marker="o", ls="", mfc="#2e9e5b", mec="k", markersize=9, label="최종 후보")]
    handles += [plt.Line2D([], [], marker="X", ls="", mfc=cmap[s], mec="k", markersize=9, label=f"탈락: {STAGE_LABEL[s]}")
                for s in STAGES if (res.drop_stage == s).any()]
    ax.legend(handles=handles, loc="upper left", fontsize=10)

    # 축 라벨 및 제목
    ax.set_xlabel("Trend β (연 성장률 %, Month-FE 제거, 95% CI)", fontsize=12, weight="bold")
    ax.set_ylabel("Residual volatility (log 잔차 SD)", fontsize=12, weight="bold")
    ax.set_title("Candidate Diagnostic Scatter (점선: 변동성 컷 robust z=1.645)", fontsize=18, weight="bold", pad=12)
    ax.tick_params(axis='both', labelsize=10)
    ax.grid(alpha=0.25)

    # ---------------------------------------------------------
    # 2. Inset (밀집 구간 확대 서브플롯)
    # ---------------------------------------------------------
    # 메인 그래프 상단 우측(폭 48%, 높이 45%) 영역에 inset 생성
    axins = inset_axes(ax, width="48%", height="45%", loc="center right", borderpad=2)

    texts_inset = []
    for _, r in res.iterrows():
        fin = r.status == "FINAL"
        c = "#2e9e5b" if fin else cmap[r.drop_stage]

        axins.errorbar(r.trend_pct_yr, r.resid_sd,
                       xerr=[[r.trend_pct_yr - r.trend_pct_ci_lo], [r.trend_pct_ci_hi - r.trend_pct_yr]],
                       fmt="none", ecolor=c, alpha=0.4, lw=1.5)
        axins.scatter(r.trend_pct_yr, r.resid_sd, s=130 if fin else 85, c=c,
                      marker="o" if fin else "X", edgecolor="k", lw=0.7, zorder=3)

        # Inset 내부 라벨
        t = axins.text(r.trend_pct_yr, r.resid_sd, str(r.code), fontsize=9.5, weight="bold" if fin else None)
        texts_inset.append(t)

    # Inset 내부 기준선 및 스타일
    axins.axhline(vol_cut, color="#7f7f7f", ls=":", lw=1.2)
    axins.grid(True, alpha=0.3, ls="--")
    axins.tick_params(axis='both', labelsize=9)

    # 확대할 X, Y 범위 (밀집 구간: Trend 2~18%, Vol 0.02~0.16)
    axins.set_xlim(2, 18)
    axins.set_ylim(0.02, 0.15)

    # adjustText 패키지가 설치되어 있으면 텍스트 자동 정렬 적용
    if HAS_ADJUST_TEXT:
        adjust_text(texts_inset, ax=axins, arrowprops=dict(arrowstyle='->', color='gray', lw=0.5))

    # 메인 그래프와 확대 영역 간 연결선 표기
    mark_inset(ax, axins, loc1=2, loc2=4, fc="none", ec="0.4", ls="--", lw=1.2)

    # 저장 및 마무리
    plt.tight_layout()
    fig.savefig(path, dpi=200)
    plt.close(fig)


# ------------------------------------------------------------------ main
def main():
    os.makedirs(OUT_DIR, exist_ok=True)
    set_font()
    df = pd.read_csv(RAW_PATH, index_col=0)
    res = compute_stats(df)
    res, flog = run_funnel(res)

    res.to_csv(f"{OUT_DIR}/screening_all_stats.csv", index=False, encoding="utf-8-sig")
    res[res.status == "DROP"][["code", "name", "drop_stage", "drop_reason", "skipped_rules"]] \
        .to_csv(f"{OUT_DIR}/screening_dropped.csv", index=False, encoding="utf-8-sig")
    res[res.status == "FINAL"].to_csv(f"{OUT_DIR}/screening_final_candidates.csv",
                                      index=False, encoding="utf-8-sig")
    flog.to_csv(f"{OUT_DIR}/screening_funnel_log.csv", index=False, encoding="utf-8-sig")
    try:
        with pd.ExcelWriter(f"{OUT_DIR}/screening_result.xlsx") as xw:
            res.to_excel(xw, sheet_name="all_stats", index=False)
            res[res.status == "FINAL"].to_excel(xw, sheet_name="final", index=False)
            res[res.status == "DROP"][["code", "name", "drop_stage", "drop_reason", "skipped_rules"]] \
                .to_excel(xw, sheet_name="dropped", index=False)
            flog.to_excel(xw, sheet_name="funnel_log", index=False)
    except Exception as e:
        print("xlsx skip:", e)

    chart_funnel(flog, len(TARGET_CODES), f"{OUT_DIR}/chart1_funnel.png")
    chart_drop(res, f"{OUT_DIR}/chart2_drop_reason.png")
    chart_scatter(res, f"{OUT_DIR}/chart3_scatter.png")
    return res, flog


if __name__ == "__main__":
    res, flog = main()
    pd.set_option("display.width", 250, "display.max_colwidth", 90, "display.max_columns", 30)
    print(flog)
    print(res[["code", "name", "status", "drop_stage", "trend_pct_yr", "trend_p",
               "trend_fdr_q", "yoy_median", "resid_sd", "r36_pct_yr", "r24_pct_yr"]].round(3))